# KDE Model Backtest

Replay the per-critic KDE model (`critic_model.py`) + `compute_edge()` against historical Kalshi prices.
Measures calibration, Brier score, and retroactive P&L across 141 resolved movies.

See `plans/plan_kde_backtest.md` for methodology.

In [ ]:
import sys, os, io, glob, contextlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# Add project root to path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from edge import compute_edge
from critic_model import (
    build_critic_profiles,
    build_kde_lambda_model,
    default_training_slugs,
    estimate_lambda,
    estimate_p_fresh,
)

PRICE_DIR = ROOT / "rt-price-histories"
THRESHOLDS = list(range(45, 100, 5))  # 45, 50, ..., 95

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
# --- Load data ---

reviews_df = pd.read_csv(ROOT / "reviews.csv")
reviews_df["estimated_timestamp"] = pd.to_datetime(
    reviews_df["estimated_timestamp"], format="ISO8601", utc=True
)

movies_df = pd.read_csv(ROOT / "movies_index.csv")
movies_df["Bet Close Date"] = pd.to_datetime(movies_df["Bet Close Date"], utc=True)

print(f"Reviews: {len(reviews_df):,} rows, {reviews_df['movie_slug'].nunique()} movies")
print(f"Movies index: {len(movies_df)} movies")

# Slugs with hourly price data
slugs_with_prices = sorted([
    d.name for d in PRICE_DIR.iterdir()
    if d.is_dir() and list(d.glob("*hour*"))
])
print(f"Slugs with hourly prices: {len(slugs_with_prices)}")

# Filter movies_df to those with prices
movies_bt = movies_df[movies_df["Slug"].isin(slugs_with_prices)].copy()
movies_bt = movies_bt.dropna(subset=["Bet Close Date"]).sort_values("Bet Close Date")
print(f"Movies for backtest: {len(movies_bt)}")

## Helper functions

In [ ]:
def load_hourly_prices(slug):
    """Load and forward-fill the hourly price CSV for a movie."""
    csv_files = list((PRICE_DIR / slug).glob("*hour*"))
    if not csv_files:
        return None
    df = pd.read_csv(csv_files[0])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    
    # Extract threshold columns
    thresh_cols = [c for c in df.columns if c.startswith("Above ")]
    
    # Forward-fill NaN prices (last traded price persists)
    df[thresh_cols] = df[thresh_cols].ffill()
    
    return df


def get_resolution(price_df):
    """Derive resolution from terminal prices. Returns {threshold: bool or None}."""
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    resolution = {}
    for col in thresh_cols:
        thresh = int(col.split()[-1])
        last_valid = price_df[col].dropna()
        if last_valid.empty:
            resolution[thresh] = None
            continue
        terminal = last_valid.iloc[-1]
        if terminal >= 90:
            resolution[thresh] = True
        elif terminal <= 10:
            resolution[thresh] = False
        else:
            resolution[thresh] = None  # ambiguous
    return resolution


def precompute_review_states(slug, reviews_df, bet_close):
    """Precompute cumulative review states sorted by timestamp.
    
    Returns a list of (timestamp, observed_critics_set, fresh_count, total_count).
    The list is sorted by timestamp. For a given snapshot_time, binary search
    to find the last entry with timestamp <= snapshot_time.
    
    Also returns the full sorted reviews for reference.
    """
    movie_reviews = reviews_df[reviews_df["movie_slug"] == slug].copy()
    # Only reviews before bet close
    movie_reviews = movie_reviews[movie_reviews["estimated_timestamp"] <= bet_close]
    movie_reviews = movie_reviews.sort_values("estimated_timestamp").reset_index(drop=True)
    
    if movie_reviews.empty:
        return [], None
    
    # Build cumulative states at each review boundary
    states = []
    critics = set()
    fresh = 0
    total = 0
    
    for _, row in movie_reviews.iterrows():
        critics = critics | {row["reviewer_name"]}
        total += 1
        if row["tomatometer_sentiment"] == "positive":
            fresh += 1
        states.append({
            "timestamp": row["estimated_timestamp"],
            "observed_critics": frozenset(critics),
            "fresh_count": fresh,
            "total_count": total,
        })
    
    return states, movie_reviews["estimated_timestamp"].iloc[0]


def get_review_state_at(states, snapshot_time):
    """Binary search for the review state at snapshot_time.
    
    Returns (observed_critics, fresh_count, total_count) or (set(), 0, 0) if no reviews yet.
    """
    if not states or snapshot_time < states[0]["timestamp"]:
        return set(), 0, 0
    
    # Binary search: find last state with timestamp <= snapshot_time
    lo, hi = 0, len(states) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if states[mid]["timestamp"] <= snapshot_time:
            lo = mid
        else:
            hi = mid - 1
    
    s = states[lo]
    return set(s["observed_critics"]), s["fresh_count"], s["total_count"]

In [ ]:
def backtest_movie(slug, reviews_df, movies_df, every_n_hours=1):
    """Run backtest for a single movie. Returns list of trade records.
    
    Args:
        every_n_hours: Evaluate every N hours by wall-clock time (1=hourly, 24=daily).
    """
    row = movies_df[movies_df["Slug"] == slug].iloc[0]
    bet_close_date = row["Bet Close Date"]  # date-only, midnight UTC
    
    # Load prices
    price_df = load_hourly_prices(slug)
    if price_df is None or price_df.empty:
        return []
    
    # Use last price CSV timestamp as effective close time.
    # Bet Close Date is date-only (midnight UTC) but markets actually close ~13-15h later.
    # The last traded timestamp is the best proxy we have.
    market_close_time = price_df["timestamp"].iloc[-1]
    
    # Resolution ground truth
    resolution = get_resolution(price_df)
    
    # Training set (no lookahead) — use date-level Bet Close Date for ordering (sufficient)
    training_slugs = default_training_slugs(
        movies_df, exclude_slug=slug, before_date=bet_close_date
    )
    if len(training_slugs) < 5:
        return []  # Too few training movies for reliable model
    
    # Build model (suppress print output)
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        profiles = build_critic_profiles(reviews_df, movies_df, training_slugs)
        model = build_kde_lambda_model(profiles)
    
    # Precompute review states (use market_close_time to include reviews up to actual close)
    review_states, first_review_ts = precompute_review_states(slug, reviews_df, market_close_time)
    
    # Subsample snapshots by actual time interval
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    
    records = []
    cached_critics, cached_fresh, cached_total = set(), 0, 0
    cached_lambda, cached_p_fresh = None, None
    last_kept_ts = None
    
    for i in range(len(price_df)):
        snap_row = price_df.iloc[i]
        snapshot_time = snap_row["timestamp"]
        
        # Time-based subsampling: keep if >= every_n_hours since last kept snapshot
        if every_n_hours > 1 and last_kept_ts is not None:
            hours_since = (snapshot_time - last_kept_ts).total_seconds() / 3600
            if hours_since < every_n_hours:
                continue
        last_kept_ts = snapshot_time
        
        hours_to_close = (market_close_time - snapshot_time).total_seconds() / 3600
        
        if hours_to_close <= 0:
            continue
        
        days_before_close = hours_to_close / 24
        
        # Get review state via binary search
        observed_critics, fresh_count, total_count = get_review_state_at(
            review_states, snapshot_time
        )
        
        # Determine if state changed (to avoid redundant p_fresh calls)
        state_changed = (total_count != cached_total)
        
        if state_changed or cached_lambda is None:
            cached_critics = observed_critics
            cached_fresh = fresh_count
            cached_total = total_count
            
            first_review_dbc = None
            if first_review_ts is not None and total_count > 0:
                first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400
            
            cached_lambda = estimate_lambda(
                model, days_before_close, hours_to_close,
                observed_critics, observed_count=total_count,
                first_review_dbc=first_review_dbc,
            )
            cached_p_fresh = estimate_p_fresh(
                profiles, observed_critics, fresh_count, total_count,
            )
        else:
            # Review state unchanged but time advanced — re-estimate lambda only
            first_review_dbc = None
            if first_review_ts is not None and cached_total > 0:
                first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400
            cached_lambda = estimate_lambda(
                model, days_before_close, hours_to_close,
                cached_critics, observed_count=cached_total,
                first_review_dbc=first_review_dbc,
            )
        
        # Evaluate each threshold
        for col in thresh_cols:
            thresh = int(col.split()[-1])
            market_price = snap_row[col]
            
            if pd.isna(market_price):
                continue  # No price yet for this threshold
            
            resolved = resolution.get(thresh)
            if resolved is None:
                continue  # Ambiguous resolution — skip
            
            try:
                result = compute_edge(
                    threshold=thresh,
                    market_price=market_price,
                    fresh_count=cached_fresh,
                    total_count=cached_total,
                    hours_to_close=hours_to_close,
                    lambda_rate=cached_lambda,
                    p_fresh=cached_p_fresh,
                )
            except (ValueError, Exception):
                continue
            
            records.append({
                "slug": slug,
                "snapshot_time": snapshot_time,
                "hours_to_close": hours_to_close,
                "threshold": thresh,
                "market_price": market_price,
                "model_p_yes": result["p_yes"],
                "edge_cents": result["edge_cents"],
                "resolved_yes": resolved,
                "lambda_rate": cached_lambda,
                "p_fresh": cached_p_fresh,
                "fresh_count": cached_fresh,
                "total_count": cached_total,
                "expected_reviews": result["expected_reviews"],
                "n_training": len(training_slugs),
            })
    
    return records

## Run backtest

Set `EVERY_N_HOURS = 24` for a quick daily-snapshot run (~2-5 min), then `1` for full hourly (~10-30 min).

In [ ]:
import time

EVERY_N_HOURS = 24  # 24=daily snapshots (~2-5 min). Set to 1 for full hourly (needs optimization).

all_records = []
slugs = movies_bt["Slug"].tolist()
skipped = []
t0 = time.time()

for idx, slug in enumerate(slugs):
    elapsed = time.time() - t0
    rate = (idx / elapsed) if elapsed > 0 and idx > 0 else 0
    eta = (len(slugs) - idx) / rate if rate > 0 else 0
    print(f"\r[{idx+1}/{len(slugs)}] {slug:<40s} ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)", end="", flush=True)
    try:
        records = backtest_movie(slug, reviews_df, movies_df, every_n_hours=EVERY_N_HOURS)
        all_records.extend(records)
    except Exception as e:
        skipped.append((slug, str(e)))

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.0f}s. {len(all_records):,} trade evaluations across {len(slugs) - len(skipped)} movies.")
if skipped:
    print(f"Skipped {len(skipped)} movies:")
    for s, err in skipped:
        print(f"  {s}: {err}")

In [ ]:
# Build DataFrame
trades = pd.DataFrame(all_records)
print(f"Shape: {trades.shape}")
print(f"Movies: {trades['slug'].nunique()}")
print(f"Snapshots: {trades.groupby('slug')['snapshot_time'].nunique().sum():,}")
print(f"\nEdge stats (all evaluations):")
print(trades["edge_cents"].describe().round(2))

## P&L calculation

In [ ]:
# Trade direction
trades["direction"] = np.where(trades["edge_cents"] >= 0, "Yes", "No")

# P&L per contract (cents)
# Buy Yes: win (100 - price) if resolved Yes, lose price if No
# Buy No:  win price if resolved No, lose (100 - price) if Yes
trades["pnl_if_yes"] = np.where(
    trades["direction"] == "Yes",
    np.where(trades["resolved_yes"], 100 - trades["market_price"], -trades["market_price"]),
    np.where(trades["resolved_yes"], -(100 - trades["market_price"]), trades["market_price"]),
)

# Time horizon buckets
trades["horizon"] = pd.cut(
    trades["hours_to_close"],
    bins=[0, 24, 72, 120, 168, float("inf")],
    labels=["T-1d", "T-3d", "T-5d", "T-7d", "T-7d+"],
    right=True,
)

# Absolute edge
trades["abs_edge"] = trades["edge_cents"].abs()

print(f"Resolution split: {trades['resolved_yes'].mean():.1%} Yes")
print(f"Direction split: {(trades['direction'] == 'Yes').mean():.1%} buy Yes")
print(f"\nHorizon distribution:")
print(trades["horizon"].value_counts().sort_index())

## Aggregate metrics by minimum edge threshold

In [ ]:
MIN_EDGES = [3, 5, 10, 15, 20]

summary_rows = []
for min_edge in MIN_EDGES:
    t = trades[trades["abs_edge"] >= min_edge].copy()
    if t.empty:
        continue
    n = len(t)
    wins = (t["pnl_if_yes"] > 0).sum()
    total_pnl = t["pnl_if_yes"].sum()
    mean_pnl = t["pnl_if_yes"].mean()
    
    summary_rows.append({
        "min_edge": min_edge,
        "trades": n,
        "win_rate": wins / n,
        "total_pnl_cents": total_pnl,
        "mean_pnl_cents": mean_pnl,
        "movies": t["slug"].nunique(),
    })

summary = pd.DataFrame(summary_rows)
print("=== P&L by minimum edge threshold ===")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print()

# Breakdown by horizon for min_edge=5
print("=== P&L by horizon (min_edge=5c) ===")
t5 = trades[trades["abs_edge"] >= 5]
for h in ["T-1d", "T-3d", "T-5d", "T-7d", "T-7d+"]:
    th = t5[t5["horizon"] == h]
    if th.empty:
        continue
    print(f"  {h}: {len(th):>6,} trades, win={th['pnl_if_yes'].gt(0).mean():.1%}, "
          f"total={th['pnl_if_yes'].sum():>+10,.0f}c, mean={th['pnl_if_yes'].mean():>+6.1f}c")

## Calibration + Brier score

In [ ]:
# Brier score
brier = ((trades["model_p_yes"] - trades["resolved_yes"].astype(float)) ** 2).mean()
print(f"Brier score (all): {brier:.4f}")

# Market implied Brier for comparison
market_brier = ((trades["market_price"] / 100 - trades["resolved_yes"].astype(float)) ** 2).mean()
print(f"Brier score (market): {market_brier:.4f}")

# By horizon
for h in ["T-1d", "T-3d", "T-5d", "T-7d", "T-7d+"]:
    th = trades[trades["horizon"] == h]
    if th.empty:
        continue
    b = ((th["model_p_yes"] - th["resolved_yes"].astype(float)) ** 2).mean()
    mb = ((th["market_price"] / 100 - th["resolved_yes"].astype(float)) ** 2).mean()
    print(f"  {h}: model={b:.4f}, market={mb:.4f}, Δ={b-mb:+.4f}")

# Calibration plot
fig, ax = plt.subplots(1, 1, figsize=(7, 7))

n_bins = 10
trades["p_bin"] = pd.cut(trades["model_p_yes"], bins=n_bins)
cal = trades.groupby("p_bin", observed=True).agg(
    pred_mean=("model_p_yes", "mean"),
    actual_mean=("resolved_yes", "mean"),
    count=("resolved_yes", "size"),
).reset_index()

ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
ax.scatter(cal["pred_mean"], cal["actual_mean"], s=cal["count"] / cal["count"].max() * 200 + 20, 
           alpha=0.7, zorder=5)
for _, r in cal.iterrows():
    ax.annotate(f"n={int(r['count']):,}", (r["pred_mean"], r["actual_mean"]),
                textcoords="offset points", xytext=(5, 5), fontsize=7)

ax.set_xlabel("Model P(Yes)")
ax.set_ylabel("Actual frequency of Yes")
ax.set_title(f"Calibration plot (Brier={brier:.4f})")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Cumulative P&L curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True)
axes = axes.flatten()

for i, min_edge in enumerate(MIN_EDGES):
    ax = axes[i]
    t = trades[trades["abs_edge"] >= min_edge].sort_values("snapshot_time").copy()
    if t.empty:
        ax.set_title(f"min_edge={min_edge}c (no trades)")
        continue
    t["cum_pnl"] = t["pnl_if_yes"].cumsum()
    ax.plot(t["snapshot_time"], t["cum_pnl"], linewidth=0.7)
    ax.axhline(0, color="k", linewidth=0.3)
    ax.set_title(f"min_edge={min_edge}c  |  {len(t):,} trades  |  {t['cum_pnl'].iloc[-1]:+,.0f}c")
    ax.set_ylabel("Cumulative P&L (cents)")

# Also plot by horizon for min_edge=5
ax = axes[5]
t5 = trades[trades["abs_edge"] >= 5].sort_values("snapshot_time").copy()
for h in ["T-1d", "T-3d", "T-5d", "T-7d"]:
    th = t5[t5["horizon"] == h].copy()
    if th.empty:
        continue
    th["cum_pnl"] = th["pnl_if_yes"].cumsum()
    ax.plot(th["snapshot_time"], th["cum_pnl"], label=h, linewidth=0.7)
ax.axhline(0, color="k", linewidth=0.3)
ax.set_title("min_edge=5c by horizon")
ax.set_ylabel("Cumulative P&L (cents)")
ax.legend(fontsize=8)

for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Edge distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution
ax = axes[0]
ax.hist(trades["edge_cents"], bins=100, alpha=0.7, edgecolor="black", linewidth=0.3)
ax.axvline(0, color="red", linewidth=1)
ax.set_xlabel("Edge (cents)")
ax.set_ylabel("Count")
ax.set_title("Edge distribution (all evaluations)")

# By horizon
ax = axes[1]
for h in ["T-1d", "T-3d", "T-5d", "T-7d"]:
    th = trades[trades["horizon"] == h]
    if th.empty:
        continue
    ax.hist(th["edge_cents"], bins=50, alpha=0.4, label=h, density=True)
ax.axvline(0, color="red", linewidth=1)
ax.set_xlabel("Edge (cents)")
ax.set_ylabel("Density")
ax.set_title("Edge distribution by horizon")
ax.legend()

plt.tight_layout()
plt.show()

# What fraction of evaluations show edge > X?
print("Fraction of evaluations with |edge| above threshold:")
for e in [3, 5, 10, 15, 20, 30]:
    frac = (trades["abs_edge"] >= e).mean()
    print(f"  |edge| >= {e:>2}c: {frac:.1%} ({int(frac * len(trades)):,} trades)")

## P&L heatmap (movie x threshold)

In [ ]:
min_edge_heatmap = 5

t_hm = trades[trades["abs_edge"] >= min_edge_heatmap]
pivot = t_hm.groupby(["slug", "threshold"])["pnl_if_yes"].sum().unstack(fill_value=0)

# Sort movies by total P&L
pivot["_total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("_total", ascending=False)
pivot = pivot.drop(columns="_total")

if not pivot.empty:
    fig, ax = plt.subplots(figsize=(12, max(8, len(pivot) * 0.25)))
    
    vmax = max(abs(pivot.values.min()), abs(pivot.values.max()), 1)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", norm=norm)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f">{c}" for c in pivot.columns], rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=6)
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Movie")
    ax.set_title(f"P&L heatmap (min_edge={min_edge_heatmap}c)")
    plt.colorbar(im, ax=ax, label="P&L (cents)", shrink=0.5)
    plt.tight_layout()
    plt.show()
    
    # Top and bottom movies
    movie_pnl = t_hm.groupby("slug")["pnl_if_yes"].agg(["sum", "count", "mean"])
    movie_pnl.columns = ["total_pnl", "trades", "mean_pnl"]
    movie_pnl = movie_pnl.sort_values("total_pnl", ascending=False)
    
    print("=== Top 10 movies ===")
    print(movie_pnl.head(10).to_string(float_format=lambda x: f"{x:.1f}"))
    print(f"\n=== Bottom 10 movies ===")
    print(movie_pnl.tail(10).to_string(float_format=lambda x: f"{x:.1f}"))
else:
    print("No trades at this min_edge threshold.")

## Diagnostics: model parameters over time

In [ ]:
# Lambda and p_fresh distributions by horizon
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for h in ["T-1d", "T-3d", "T-5d", "T-7d"]:
    th = trades[trades["horizon"] == h]
    if th.empty:
        continue
    axes[0].hist(th["lambda_rate"], bins=50, alpha=0.4, label=h, density=True)
    axes[1].hist(th["p_fresh"], bins=50, alpha=0.4, label=h, density=True)

axes[0].set_xlabel("Lambda (reviews/hr)")
axes[0].set_title("Lambda distribution by horizon")
axes[0].legend()
axes[1].set_xlabel("p_fresh")
axes[1].set_title("p_fresh distribution by horizon")
axes[1].legend()
plt.tight_layout()
plt.show()

# Ambiguous resolutions
print("\n=== Resolution ambiguity check ===")
for slug in trades["slug"].unique()[:5]:  # spot check
    pdf = load_hourly_prices(slug)
    res = get_resolution(pdf)
    ambig = {k: v for k, v in res.items() if v is None}
    if ambig:
        print(f"  {slug}: ambiguous thresholds = {list(ambig.keys())}")

n_ambig_thresh = sum(
    1 for slug in trades["slug"].unique()
    for v in get_resolution(load_hourly_prices(slug)).values()
    if v is None
)
print(f"\nTotal ambiguous threshold-resolutions: {n_ambig_thresh}")

## P&L by edge magnitude bucket

In [ ]:
# P&L by edge magnitude bucket
trades["edge_bucket"] = pd.cut(
    trades["abs_edge"],
    bins=[0, 3, 5, 10, 15, 20, 30, 50, 100],
    labels=["0-3", "3-5", "5-10", "10-15", "15-20", "20-30", "30-50", "50+"],
    right=True,
)

edge_pnl = trades.groupby("edge_bucket", observed=True).agg(
    trades=("pnl_if_yes", "size"),
    win_rate=("pnl_if_yes", lambda x: (x > 0).mean()),
    total_pnl=("pnl_if_yes", "sum"),
    mean_pnl=("pnl_if_yes", "mean"),
    mean_edge=("abs_edge", "mean"),
).reset_index()

print("=== P&L by edge magnitude ===")
print(edge_pnl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# P&L by direction
print("\n=== P&L by direction (min_edge=5c) ===")
t5 = trades[trades["abs_edge"] >= 5]
for d in ["Yes", "No"]:
    td = t5[t5["direction"] == d]
    if td.empty:
        continue
    print(f"  Buy {d}: {len(td):,} trades, win={td['pnl_if_yes'].gt(0).mean():.1%}, "
          f"total={td['pnl_if_yes'].sum():+,.0f}c, mean={td['pnl_if_yes'].mean():+.1f}c")

## Position-level analysis (No-only strategy)

De-duplicate to one entry per (movie, threshold). Enter the **first** time the No edge exceeds the threshold in the T-5d to T-1d window, hold to resolution.

In [ ]:
# Dataset time span for annualization
earliest_close = movies_bt["Bet Close Date"].min()
latest_close = movies_bt["Bet Close Date"].max()
span_days = (latest_close - earliest_close).days
span_years = span_days / 365.25
print(f"Dataset spans {earliest_close.date()} to {latest_close.date()} = {span_days} days ({span_years:.2f} years)")
print(f"Movies: {len(movies_bt)}, ~{len(movies_bt)/span_years:.0f} movies/year")
print()

ACTION_WINDOW = (24, 120)  # T-5d to T-1d in hours

for min_edge in [5, 10, 15, 20]:
    # Filter: No direction, sufficient edge, action window
    mask = (
        (trades["direction"] == "No") &
        (trades["abs_edge"] >= min_edge) &
        (trades["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades["hours_to_close"] <= ACTION_WINDOW[1])
    )
    no_trades = trades[mask].copy()
    
    if no_trades.empty:
        print(f"min_edge={min_edge}c: no trades in action window")
        continue
    
    # De-duplicate: first entry per (movie, threshold) — simulates enter-once-and-hold
    no_trades = no_trades.sort_values("snapshot_time")
    positions = no_trades.groupby(["slug", "threshold"]).first().reset_index()
    
    # Position-level P&L
    # Buy No costs (100 - market_price) per contract
    # If resolved No: profit = market_price
    # If resolved Yes: loss = -(100 - market_price)
    positions["entry_cost"] = 100 - positions["market_price"]
    positions["pos_pnl"] = np.where(
        ~positions["resolved_yes"],          # resolved No = we win
        positions["market_price"],            # profit
        -positions["entry_cost"],             # loss
    )
    
    n_positions = len(positions)
    n_movies = positions["slug"].nunique()
    positions_per_movie = n_positions / n_movies
    win_rate = (positions["pos_pnl"] > 0).mean()
    avg_entry_cost = positions["entry_cost"].mean()
    avg_pnl = positions["pos_pnl"].mean()
    total_pnl = positions["pos_pnl"].sum()
    total_capital = positions["entry_cost"].sum()
    roi = total_pnl / total_capital if total_capital > 0 else 0
    
    # Per-movie stats
    movie_stats = positions.groupby("slug").agg(
        positions=("pos_pnl", "count"),
        capital=("entry_cost", "sum"),
        pnl=("pos_pnl", "sum"),
    )
    avg_capital_per_movie = movie_stats["capital"].mean()
    avg_pnl_per_movie = movie_stats["pnl"].mean()
    
    # Annualize
    annual_movies = n_movies / span_years
    annual_pnl = total_pnl / span_years
    
    print(f"=== No-only, min_edge={min_edge}c, T-5d to T-1d ===")
    print(f"  Positions: {n_positions} across {n_movies} movies ({positions_per_movie:.1f} per movie)")
    print(f"  Win rate: {win_rate:.1%}")
    print(f"  Avg entry cost: {avg_entry_cost:.1f}c per contract")
    print(f"  Avg P&L per position: {avg_pnl:+.1f}c")
    print(f"  Total P&L: {total_pnl:+,.0f}c  (${total_pnl/100:+,.2f} at $1/contract)")
    print(f"  Total capital deployed: {total_capital:,.0f}c  (${total_capital/100:,.2f})")
    print(f"  ROI: {roi:.1%}")
    print(f"  Per movie avg: {avg_capital_per_movie:.0f}c risk, {avg_pnl_per_movie:+.0f}c profit ({positions_per_movie:.1f} positions)")
    print(f"  Annualized (~{annual_movies:.0f} movies/yr): ~{annual_pnl:+,.0f}c/yr (${annual_pnl/100:+,.0f}/yr at $1/contract)")
    print()

In [ ]:
# Per-movie P&L distribution for No-only, min_edge=10c, T-5d to T-1d
min_edge_rec = 10

mask = (
    (trades["direction"] == "No") &
    (trades["abs_edge"] >= min_edge_rec) &
    (trades["hours_to_close"] >= 24) &
    (trades["hours_to_close"] <= 120)
)
no_trades = trades[mask].sort_values("snapshot_time")
positions = no_trades.groupby(["slug", "threshold"]).first().reset_index()
positions["entry_cost"] = 100 - positions["market_price"]
positions["pos_pnl"] = np.where(
    ~positions["resolved_yes"],
    positions["market_price"],
    -positions["entry_cost"],
)

movie_pnl = positions.groupby("slug").agg(
    n_positions=("pos_pnl", "count"),
    total_pnl=("pos_pnl", "sum"),
    capital=("entry_cost", "sum"),
    win_rate=("pos_pnl", lambda x: (x > 0).mean()),
).sort_values("total_pnl", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(movie_pnl["total_pnl"], bins=30, alpha=0.7, edgecolor="black", linewidth=0.3)
ax.axvline(0, color="red", linewidth=1)
ax.axvline(movie_pnl["total_pnl"].mean(), color="blue", linewidth=1, linestyle="--",
           label=f'mean={movie_pnl["total_pnl"].mean():.0f}c')
ax.axvline(movie_pnl["total_pnl"].median(), color="green", linewidth=1, linestyle="--",
           label=f'median={movie_pnl["total_pnl"].median():.0f}c')
ax.set_xlabel("P&L per movie (cents, 1 contract/position)")
ax.set_ylabel("Count")
ax.set_title(f"Per-movie P&L distribution (No-only, min_edge={min_edge_rec}c)")
ax.legend()

ax = axes[1]
ax.hist(movie_pnl["n_positions"], bins=range(0, movie_pnl["n_positions"].max() + 2),
        alpha=0.7, edgecolor="black", linewidth=0.3)
ax.set_xlabel("Positions per movie")
ax.set_ylabel("Count")
ax.set_title("Number of No positions entered per movie")

plt.tight_layout()
plt.show()

print(f"Movies with at least 1 position: {len(movie_pnl)}")
print(f"Movies with positive P&L: {(movie_pnl['total_pnl'] > 0).sum()} ({(movie_pnl['total_pnl'] > 0).mean():.0%})")
print(f"\nPer-movie P&L stats (cents):")
print(movie_pnl["total_pnl"].describe().round(1))
print(f"\nMedian positions per movie: {movie_pnl['n_positions'].median():.0f}")
print(f"Mean capital per movie: {movie_pnl['capital'].mean():.0f}c (${movie_pnl['capital'].mean()/100:.2f})")

## Findings

Run the notebook, then write results to `findings/kde_backtest.md`.